# Training of Generative Models on the German Credit Risk Dataset

## Introduction

This notebook constitutes the starting point of the training phase within our comprehensive study on the evaluation of generative models for tabular data. Its primary objective is to initiate the training of four selected models—CopulaGAN, TVAE, CTGAN, and GReaT (implemented with the distil-GPT2 backbone)—on the German Credit Risk dataset. The synthetic data produced in this phase will serve as the foundation for subsequent analyses, where quality, fidelity, and utility will be systematically assessed using multiple evaluation metrics.

As this is the first notebook in the workflow, it also serves as a reference point for the recommended reading order. To ensure clarity and avoid unnecessary repetition, the notebooks should be consulted in the following sequence:

Training of Generative Models on the German Credit Risk Dataset

Training of GReaT (GPT-2) Model on the German Credit Risk Dataset

Synthetic Data Generation on the German Credit Risk Dataset

Training of Generative Models on the Australian Credit Approval Dataset

Training of GReaT (GPT-2) Model on the Australian Credit Approval Dataset

Synthetic Data Generation on the Australian Credit Approval Dataset

Final Evaluation of Synthetic Data

This structure facilitates consistency across the analysis: the present notebook introduces the core training methodology, while subsequent notebooks build upon it to cover dataset-specific generation processes and, finally, the comprehensive evaluation of synthetic data.

## 0 Imports and function definition

This notebook uses a set of external libraries that support data handling, model training, and persistence of trained objects. Specifically, pandas is employed for tabular data manipulation, while the SDV library provides the TVAESynthesizer, CTGANSynthesizer, and CopulaGANSynthesizer classes for generative modeling. SingleTableMetadata is used to represent dataset metadata, and the be_great package offers the GReaT model, a transformer-based approach adapted for tabular data. Finally, pickle is used for saving and loading trained models.

The precise use of each component will be demonstrated in the relevant sections of this notebook, showing how they integrate into the overall workflow.

In [ ]:
!pip install be_great peft==0.9.0 sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import os
from be_great import GReaT
from sdv.single_table import TVAESynthesizer, CTGANSynthesizer, CopulaGANSynthesizer
from sdv.metadata import SingleTableMetadata
import pickle

In [ ]:
os.environ["WANDB_DISABLED"] = "true" #to avoid the requests from WANDB

The function preprocess_dataset is designed to prepare tabular data for subsequent modeling tasks by standardizing data types and addressing missing values in both categorical and numerical features. It operates on a copy of the original DataFrame to ensure that the input data remains unaltered, thereby supporting reproducibility and preventing unintended side effects.

First, all columns specified as categorical are explicitly cast to the object type, while numerical features are cast to float. This ensures type consistency across the dataset, which is particularly important for downstream pipelines that rely on strict type definitions for preprocessing and modeling.

Second, the function addresses missing values in a systematic way. For categorical variables, missing entries are replaced with the placeholder label "Unknown", preventing the introduction of biases while maintaining the categorical nature of the data. For numerical variables, missing values are imputed with the mean of the corresponding feature, a simple yet widely adopted approach that preserves the overall distribution of the data.

The resulting DataFrame provides a standardized and fully imputed version of the dataset, ensuring compatibility with preprocessing pipelines and generative models implemented later in the study.

In [ ]:
def preprocess_dataset(df, target_column, categorical_features, numerical_features):
    df = df.copy()

    for col in categorical_features:
        df[col] = df[col].astype('object')

    for col in numerical_features:
        df[col] = df[col].astype('float')

    for col in categorical_features:
        df[col] = df[col].fillna('Unknown')

    for col in numerical_features:
        df[col] = df[col].fillna(df[col].mean())

    return df


A set of custom functions has been implemented to streamline the training and persistence of the selected generative models. Each function follows a consistent structure: the model is instantiated, trained on the German Credit Risk dataset, and subsequently stored in serialized form using the pickle library. This modular design not only improves code readability but also ensures that each trained model can be reused in later stages of the workflow without requiring retraining.

For CopulaGAN, TVAE, and CTGAN, the functions first construct a SingleTableMetadata object, which automatically infers schema information from the input dataset. The corresponding synthesizer class from the SDV library is then initialized and fitted to the data. Once training is complete, the fitted synthesizer is serialized into a dedicated file (.pkl) that uniquely identifies the model.

In the case of GReaT, the function initializes a transformer-based model built on the distil-GPT2 architecture. Training hyperparameters such as batch_size, epochs, and the efficient_finetuning option can be specified to control resource usage and training efficiency. Similar to the other models, the trained instance is stored in a serialized format for subsequent use in the synthetic data generation phase.

By encapsulating the training and saving process into dedicated functions, this approach promotes consistency, modularity, and reproducibility across experiments, ensuring that models can be seamlessly integrated into the broader evaluation framework.

In [ ]:
def train_and_save_model_CopulaGAN(df):

  filename = "copulagan_german.pkl"

  metadata = SingleTableMetadata()
  metadata.detect_from_dataframe(data=df)
  synthesizer = CopulaGANSynthesizer(metadata)
  synthesizer.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
      pickle.dump(synthesizer, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def train_and_save_model_TVAE(df):

  filename = "tvae_german.pkl"

  metadata = SingleTableMetadata()
  metadata.detect_from_dataframe(data=df)
  synthesizer = TVAESynthesizer(metadata)
  synthesizer.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
      pickle.dump(synthesizer, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def train_and_save_model_CTGAN(df):

  filename = "ctgan_german.pkl"

  metadata = SingleTableMetadata()
  metadata.detect_from_dataframe(data=df)
  synthesizer = CTGANSynthesizer(metadata)
  synthesizer.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
    pickle.dump(synthesizer, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def train_and_save_model_distil_GReaT(df, batch_size, epochs, efficient_finetuning):

  filename = "distil_GReaT_german.pkl"

  model = GReaT(llm="distilgpt2", batch_size=batch_size, epochs=epochs, fp16=True, efficient_finetuning=efficient_finetuning)
  model.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
    pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)

## 1 Dataset loading

The German Credit Risk dataset is first loaded from its CSV file into a pandas DataFrame using the read_csv function. This dataset, widely used in credit risk modeling research, provides the foundation for training and evaluating the generative models implemented in this study. The original data can be accessed from the following source: Kaggle - https://www.kaggle.com/datasets/uciml/german-credit.

In [ ]:
german_df = pd.read_csv('german_credit_risk.csv')

Based on the dataset documentation, we identify the categorical and numerical features of the German Credit Risk dataset. The first column, serving as an index, is removed to simplify further processing. Categorical features include demographic and financial attributes, while numerical features capture continuous variables such as age, credit amount, and duration.

The dataset is then preprocessed using the preprocess_dataset function, which standardizes data types and imputes missing values, producing a clean and structured version ready for model training.

In [ ]:
german_df = german_df.drop(german_df.columns[0], axis=1)

german_categorical_features = ['Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']
german_numerical_features = ['Age', 'Credit amount', 'Duration']

In [ ]:
preprocessed_dataset = preprocess_dataset(german_df, 'Risk', german_categorical_features, german_numerical_features)

/tmp/ipython-input-4-3949366936.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna('Unknown')


In [ ]:
preprocessed_dataset.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67.0,male,2,own,Unknown,little,1169.0,6.0,radio/TV,good
1,22.0,female,2,own,little,moderate,5951.0,48.0,radio/TV,bad
2,49.0,male,1,own,little,Unknown,2096.0,12.0,education,good
3,45.0,male,2,free,little,little,7882.0,42.0,furniture/equipment,good
4,53.0,male,2,free,little,little,4870.0,24.0,car,bad


## 2 Train

Following the preprocessing step, we proceed to the training of the generative models using the custom functions defined earlier. Each function encapsulates the complete workflow for a specific model, including instantiation, fitting to the preprocessed dataset, and serialization of the trained model. This modular approach allows for a streamlined execution of multiple models while maintaining reproducibility and consistency across experiments.

### CopulaGAN

In [ ]:
train_and_save_model_CopulaGAN(preprocessed_dataset)

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


### TVAE

In [ ]:
train_and_save_model_TVAE(preprocessed_dataset)

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


### CTGAN

In [ ]:
train_and_save_model_CTGAN(preprocessed_dataset)

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


### Distil-GReaT

In [ ]:
train_and_save_model_distil_GReaT(preprocessed_dataset, 32, 200, "lora")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:145: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:861: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/be_great/great.py:174: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `GReaTTrainer.__init__`. Use `processing_class` instead.
  great_trainer = GReaTTrainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 294,912 || all params: 82,207,488 || trainable%: 0.35874104315168953


You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,2.383300
1000,1.185100
1500,1.094400
2000,1.050700
2500,1.018600
3000,0.991300
3500,0.969400
4000,0.948600
4500,0.933000
5000,0.921900
